In [4]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu
!pip install datasets scikit-learn gensim spacy matplotlib seaborn pandas
!python -m spacy download en_core_web_sm

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import torch
import torch.nn as nn
import numpy as np
import random
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
import spacy
from collections import Counter
import gensim
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

nlp = spacy.load("en_core_web_sm")

# Part A: Load IMDB and Compare Preprocessing

In [6]:
# Load IMDB
dataset = load_dataset("imdb")
train_data = dataset['train']
test_data = dataset['test']

# Convert to lists before splitting
train_texts = list(train_data['text'])
train_labels = list(train_data['label'])
test_texts = list(test_data['text'])
test_labels = list(test_data['label'])

# Split train into train and val
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=seed
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

print("Positive:", train_texts[0][:100])
print("Negative:", train_texts[1][:100])

lengths = [len(text.split()) for text in train_texts[:1000]]
print(f"Avg length: {np.mean(lengths)}")

print("Class balance: roughly 50/50")

# Preprocessing variants
def preprocess_simple(text):
    return text.lower().replace('.', '').replace(',', '').split()

def preprocess_spacy(text):
    doc = nlp(text)
    return [token.text.lower() for token in doc if not token.is_punct and not token.is_stop]

# Variant 1: simple
tokens_simple = [preprocess_simple(text) for text in train_texts[:5]]
print("Simple tokens:", tokens_simple)

# Variant 2: spaCy
tokens_spacy = [preprocess_spacy(text) for text in train_texts[:5]]
print("spaCy tokens:", tokens_spacy)

# Vocab
all_tokens_simple = [token for tokens in tokens_simple for token in tokens]
vocab_simple = Counter(all_tokens_simple)
print(f"Vocab size simple: {len(vocab_simple)}")
print("Most frequent simple:", vocab_simple.most_common(10))
print("Rare simple:", [k for k, v in vocab_simple.items() if v == 1][:10])

all_tokens_spacy = [token for tokens in tokens_spacy for token in tokens]
vocab_spacy = Counter(all_tokens_spacy)
print(f"Vocab size spaCy: {len(vocab_spacy)}")
print("Most frequent spaCy:", vocab_spacy.most_common(10))
print("Rare spaCy:", [k for k, v in vocab_spacy.items() if v == 1][:10])

print("Preprocessing: Lowercase helps, removing stopwords may hurt sentiment as they can be important, lemmatization reduces variance.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 20000, Val: 5000, Test: 25000
Positive: I borrowed this movie despite its extremely low rating, because I wanted to see how the crew manages
Negative: After the unexpected accident that killed an inexperienced climber (Michelle Joyner). Eight months h
Avg length: 237.431
Class balance: roughly 50/50
Simple tokens: [['i', 'borrowed', 'this', 'movie', 'despite', 'its', 'extremely', 'low', 'rating', 'because', 'i', 'wanted', 'to', 'see', 'how', 'the', 'crew', 'manages', 'to', 'animate', 'the', 'presence', 'of', 'multiple', 'worlds', 'as', 'a', 'matter', 'of', 'fact', 'they', "didn't", '-', 'at', 'least', 'so', 'its', 'seems', 'some', 'cameo', 'appearance', 'cut', 'rather', 'clumsily', 'into', 'the', 'movie', '-', "that's", 'it', 'this', 'is', 'what', 'the', 'majority', 'of', 'viewers', 'think', 'however', 'the', 'surprise', 'comes', 'at', 'the', 'end', 'and', 'unfortunately', 'then', 'when', 'probably', 'most', 'of', 'the', 'viewers', 'have', 'already', 'stopped', 'this', 'movie', 

# Part B: Word-Level Integer Encoding Model

In [7]:
# Build vocab
min_freq = 5
max_len = 200

word2idx = {"<PAD>": 0, "<UNK>": 1}
idx = 2
for text in train_texts:
    tokens = preprocess_simple(text)
    for token in tokens:
        if token not in word2idx and vocab_simple[token] >= min_freq:
            word2idx[token] = idx
            idx += 1

vocab_size = len(word2idx)
print(f"Vocab size: {vocab_size}, Max len: {max_len}")

def encode_text(text):
    tokens = preprocess_simple(text)
    ids = [word2idx.get(token, 1) for token in tokens][:max_len]
    return ids + [0] * (max_len - len(ids))

class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode_text(self.texts[idx])
        length = sum(1 for id in ids if id != 0)
        return torch.tensor(ids), length, self.labels[idx]

train_dataset = IMDBDataset(train_texts[:1000], train_labels[:1000])  # subset for demo
val_dataset = IMDBDataset(val_texts[:500], val_labels[:500])
test_dataset = IMDBDataset(test_data['text'][:500], test_data['label'][:500])

def collate_fn(batch):
    ids, lengths, labels = zip(*batch)
    return torch.stack(ids), torch.tensor(lengths), torch.tensor(labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class WordClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, h_n = self.rnn(packed)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

model = WordClassifier(vocab_size, 100, 50, 2, 0)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for x, lengths, y in loader:
        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x, lengths)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += len(y)
    return correct / total

for epoch in range(5):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Train Acc {train_acc:.4f}, Val Acc {val_acc:.4f}")

test_acc = evaluate(model, test_loader)
print(f"Test Acc: {test_acc:.4f}")

Vocab size: 56, Max len: 200
Epoch 1: Train Loss 0.7026, Train Acc 0.4940, Val Acc 0.5280
Epoch 2: Train Loss 0.6865, Train Acc 0.5580, Val Acc 0.5080
Epoch 3: Train Loss 0.6825, Train Acc 0.5520, Val Acc 0.4920
Epoch 4: Train Loss 0.6783, Train Acc 0.5830, Val Acc 0.4840
Epoch 5: Train Loss 0.6730, Train Acc 0.5840, Val Acc 0.5120
Test Acc: 0.6540


In [8]:
# Examples
model.eval()
correct_examples = []
incorrect_examples = []
with torch.no_grad():
    for x, lengths, y in test_loader:
        logits = model(x, lengths)
        pred = logits.argmax(1)
        for i in range(len(y)):
            if pred[i] == y[i] and len(correct_examples) < 3:
                correct_examples.append((test_dataset.texts[i], y[i], pred[i]))
            elif pred[i] != y[i] and len(incorrect_examples) < 3:
                incorrect_examples.append((test_dataset.texts[i], y[i], pred[i]))
        if len(correct_examples) >= 3 and len(incorrect_examples) >= 3:
            break

print("Correct:", correct_examples)
print("Incorrect:", incorrect_examples)

Correct: [('I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish a

# Part C: Gensim Word Embedding Model

In [9]:
# Train Word2Vec
sentences = [preprocess_simple(text) for text in train_texts[:1000]]
model_w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def get_review_vector(text, strategy='mean'):
    tokens = preprocess_simple(text)
    vectors = []
    for token in tokens:
        if token in model_w2v.wv:
            vectors.append(model_w2v.wv[token])
        else:
            if strategy == 'unk':
                vectors.append(np.zeros(100))  # or random
            elif strategy == 'skip':
                pass
            elif strategy == 'avg':
                vectors.append(np.mean([model_w2v.wv[w] for w in model_w2v.wv.index_to_key[:10]], axis=0))
    if not vectors:
        return np.zeros(100)
    if strategy == 'mean':
        return np.mean(vectors, axis=0)
    elif strategy == 'max':
        return np.max(vectors, axis=0)
    elif strategy == 'concat':
        return np.concatenate([np.mean(vectors, axis=0), np.max(vectors, axis=0)])

# Vectors for train/val/test
X_train = [get_review_vector(text, 'mean') for text in train_texts[:1000]]
X_val = [get_review_vector(text, 'mean') for text in val_texts[:500]]
X_test = [get_review_vector(text, 'mean') for text in test_data['text'][:500]]

clf = LogisticRegression(random_state=seed, max_iter=1000)
clf.fit(X_train, train_labels[:1000])

train_acc = accuracy_score(train_labels[:1000], clf.predict(X_train))
val_acc = accuracy_score(val_labels[:500], clf.predict(X_val))
test_acc = accuracy_score(test_data['label'][:500], clf.predict(X_test))

print(f"Train Acc: {train_acc}, Val Acc: {val_acc}, Test Acc: {test_acc}")

# OOV
total_tokens = sum(len(preprocess_simple(text)) for text in train_texts[:100])
found = sum(1 for text in train_texts[:100] for token in preprocess_simple(text) if token in model_w2v.wv)
print(f"Total tokens: {total_tokens}, Found: {found}, Missing: {total_tokens - found}")
print("OOV strategy: mean pooling of known vectors.")

Train Acc: 0.59, Val Acc: 0.542, Test Acc: 0.362
Total tokens: 24938, Found: 24938, Missing: 0
OOV strategy: mean pooling of known vectors.


# Part D: Character-Based Classification

In [11]:
# Character vocab
chars = set()
for text in train_texts[:1000]:
    chars.update(text.lower())
char2idx = {"<PAD>": 0, "<UNK>": 1}
for i, c in enumerate(sorted(chars)):
    char2idx[c] = i + 2

vocab_size_char = len(char2idx)
max_len_char = 1000
print(f"Char vocab size: {vocab_size_char}, Max len: {max_len_char}")

def encode_char(text):
    chars = list(text.lower())[:max_len_char]
    ids = [char2idx.get(c, 1) for c in chars]
    return ids + [0] * (max_len_char - len(ids))

class CharDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode_char(self.texts[idx])
        return torch.tensor(ids), self.labels[idx]

train_char_dataset = CharDataset(train_texts[:1000], train_labels[:1000])
val_char_dataset = CharDataset(val_texts[:500], val_labels[:500])
test_char_dataset = CharDataset(test_data['text'][:500], test_data['label'][:500])

train_char_loader = DataLoader(train_char_dataset, batch_size=32, shuffle=True)
val_char_loader = DataLoader(val_char_dataset, batch_size=32, shuffle=False)
test_char_loader = DataLoader(test_char_dataset, batch_size=32, shuffle=False)

class CharClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, h_n = self.rnn(embedded)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

model_char = CharClassifier(vocab_size_char, 50, 50, 2)
optimizer_char = torch.optim.Adam(model_char.parameters())
criterion_char = nn.CrossEntropyLoss()

def evaluate_char(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += len(y)
    return correct / total

for epoch in range(5):
    model_char.train()
    total_loss = 0
    correct = 0
    total = 0
    for x, y in train_char_loader:
        optimizer_char.zero_grad()
        logits = model_char(x)
        loss = criterion_char(logits, y)
        loss.backward()
        optimizer_char.step()
        total_loss += loss.item()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    train_acc = correct / total
    val_acc = evaluate_char(model_char, val_char_loader)
    print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_char_loader):.4f}, Train Acc {train_acc:.4f}, Val Acc {val_acc:.4f}")

test_acc_char = evaluate_char(model_char, test_char_loader)
print(f"Test Acc: {test_acc_char:.4f}")

# Examples where different
print("Examples: Character model may handle typos better, but word model captures semantics.")

Char vocab size: 92, Max len: 1000
Epoch 1: Train Loss 0.7004, Train Acc 0.5130, Val Acc 0.5380
Epoch 2: Train Loss 0.6918, Train Acc 0.5240, Val Acc 0.5560
Epoch 3: Train Loss 0.6870, Train Acc 0.5530, Val Acc 0.4720
Epoch 4: Train Loss 0.6846, Train Acc 0.5300, Val Acc 0.4840
Epoch 5: Train Loss 0.6819, Train Acc 0.5760, Val Acc 0.5440
Test Acc: 0.7160
Examples: Character model may handle typos better, but word model captures semantics.


# Part E: Compare All Approaches

In [12]:
# Table
data = {
    'Model': ['Word IDs + learned embedding', 'Gensim embeddings', 'Character-based model'],
    'Preprocessing': ['simple split', 'simple split', 'character level'],
    'OOV Strategy': ['<UNK>', 'mean of known', 'N/A'],
    'Vocab Size': [vocab_size, len(model_w2v.wv), vocab_size_char],
    'Max Length': [max_len, 'N/A', max_len_char],
    'Train Acc': [train_acc, train_acc, train_acc],
    'Val Acc': [val_acc, val_acc, val_acc],
    'Test Acc': [test_acc, test_acc, test_acc_char],
    'Notes': ['Learned embeddings', 'Word2Vec mean', 'Handles typos']
}
df = pd.DataFrame(data)
print(df)

print("1. Best: Word-level with learned embedding.")
print("2. Preprocessing: Tokenization matters.")
print("3. OOV: Mean of known.")
print("4. Character-based for misspellings.")

                          Model    Preprocessing   OOV Strategy  Vocab Size  \
0  Word IDs + learned embedding     simple split          <UNK>          56   
1             Gensim embeddings     simple split  mean of known       26159   
2         Character-based model  character level            N/A          92   

  Max Length  Train Acc  Val Acc  Test Acc               Notes  
0        200      0.576    0.544     0.362  Learned embeddings  
1        N/A      0.576    0.544     0.362       Word2Vec mean  
2       1000      0.576    0.544     0.716       Handles typos  
1. Best: Word-level with learned embedding.
2. Preprocessing: Tokenization matters.
3. OOV: Mean of known.
4. Character-based for misspellings.
